# Netflix Content Analytics & Recommendation
## Business Context

Netflix operates a large and diverse content catalog consisting of movies and TV shows across different genres, countries, release periods, and content types. As the catalog expands, effectively understanding and utilizing this content data becomes increasingly important.

Two areas of opportunity are particularly relevant: improving how users discover relevant content and generating data-driven insights into the composition and evolution of the content catalog.

This analysis uses Netflix catalog data to examine content patterns, develop a content-based recommendation approach, identify historical trends, forecast future content growth, and generate insights that could support content discovery and strategic decision-making.

## Business Objective

The objective of this analysis is to transform Netflix catalog data into actionable insights that can help answer questions such as:

What does Netflix's content catalog look like across different markets, genres, and time periods?
What patterns and trends can be identified in the growth and composition of the catalog?
How can available content information be used to identify titles that are similar to one another?
What future content trends can be anticipated from historical patterns?
What predictive insights can be derived from the available content attributes?

The analysis will ultimately connect these findings to potential opportunities to improve content discovery and support data-driven decisions regarding Netflix's content portfolio.

## Data Understanding

The project uses a Netflix titles dataset containing metadata for movies and TV shows available on the platform. The dataset provides information about the content itself, its production details, catalog addition date, release year, rating, duration, and content categories.

### Dataset Overview

The dataset initially contained **8,790 records and 10 variables**. Each row represents a Netflix title, while the columns describe different attributes of that title.

| Attribute                    | Description                           |
| ---------------------------- | ------------------------------------- |
| **Rows**                     | 8,790                                 |
| **Columns**                  | 10                                    |
| **Content types**            | Movies and TV Shows                   |
| **Time-related information** | Release year and date added           |
| **Categorical information**  | Type, rating, country, and categories |
| **Content information**      | Title, director, and duration         |

### Variables and Data Types

| Variable       | Data Type | Description                                                     |
| -------------- | --------- | --------------------------------------------------------------- |
| `show_id`      | Object    | Unique identifier for each title                                |
| `type`         | Object    | Indicates whether the title is a Movie or TV Show               |
| `title`        | Object    | Name of the title                                               |
| `director`     | Object    | Director associated with the title                              |
| `country`      | Object    | Country or countries associated with the title                  |
| `date_added`   | Datetime  | Date the title was added to the Netflix catalog                 |
| `release_year` | Integer   | Year the title was released                                     |
| `rating`       | Object    | Content rating assigned to the title                            |
| `duration`     | Object    | Runtime in minutes for movies or number of seasons for TV shows |
| `listed_in`    | Object    | Categories or genres associated with the title                  |

### Understanding the Data Structure

The dataset combines **categorical, textual, numerical, and date-related information**. Some fields, such as `country` and `listed_in`, contain multiple values within a single record, while `duration` uses different units depending on the content type. These characteristics are important when determining how the data should be cleaned, transformed, and later represented for analysis and machine learning.

The dataset therefore provides several dimensions for analysis, including content distribution, production geography, genre patterns, release trends, content characteristics, and relationships between different title attributes. This understanding informed the subsequent data-quality assessmens addressed throughout the analysis.

## Data Preparation

Reliable analysis depends on reliable data. Before investigating trends or developing predictive models, the Netflix dataset will be assessed and prepared to ensure that the information used throughout the analysis is sufficiently complete, consistent, and suitable for the intended analytical applications.

### The data preparation process will focus on:

Understanding the structure and characteristics of the available data
Identifying missing, duplicated, inconsistent, or potentially invalid records
Standardizing variables and categorical values where necessary
Creating features that provide additional analytical value
Validating the resulting dataset before downstream analysis

The objective is not simply to clean the dataset, but to establish a reliable analytical foundation for the business questions addressed throughout the analysis.

### 1. Data Loading & Initial Inspection

Before assessing and improving data quality, the available Netflix content data must first be examined to understand its structure, coverage, and characteristics.

This initial inspection establishes a baseline for the raw dataset and helps identify potential issues that may affect subsequent analysis and modeling.

The assessment will focus on:

* Dataset dimensions and structure
* Available variables and their data types
* Missing values
* Duplicate records
* Unique and categorical values
* Basic descriptive statistics

The findings from this stage will guide the subsequent data cleaning and feature engineering decisions.

In [1]:
# Importing libraries
import pandas as pd
import numpy as np 


In [3]:
# Loading our dataset
netflix_df = pd.read_csv("Dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset.csv'

In [ ]:
netflix_df.head()

In [ ]:
# Convert valid date strings into pandas datetime values
netflix_df["date_added"] = pd.to_datetime(netflix_df["date_added"])

In [ ]:
# Confirm the updated data type and check for missing dates
netflix_df["date_added"].dtype, netflix_df["date_added"].isna().sum()

In [ ]:
# analyzing the number of records and columns in our dataset
netflix_df.shape

In [ ]:
# checking the data types contained in different columns of our dataset
netflix_df.dtypes

In [ ]:
netflix_df.info

In [ ]:
netflix_df. describe(include = "all")

In [ ]:
# investigating nulls
netflix_df.isnull().sum()

In [ ]:
netflix_df.duplicated().sum()

#### Observation
After some investigation and data quality checks, I just found out that this dataset 8790 rows and 10 columns. It does not have any record of duplicated records and no nulls as well.

### 2. Data quality checks
I will be checking the formatting and consistency in naming and and also have the date_added column formated to datetime.

In [ ]:
# Compare unique show IDs with the total number of records
netflix_df["show_id"].nunique(), netflix_df.shape[0]

In [ ]:
# Count each content type, including missing values
netflix_df["type"].value_counts(dropna=False)

In [ ]:
# Count each rating category, including missing values
netflix_df["rating"].value_counts(dropna=False)

In [ ]:
# Display all unique rating values
netflix_df["rating"].unique()

**Finding:** The rating field contains 14 distinct rating categories with varying frequencies. The values appear consistently formatted, with no immediate evidence of capitalization or whitespace inconsistencies. Both TV and film rating systems are represented, which is consistent with the dataset containing multiple content types.

In [ ]:
# Inspect the first few date values and their current representation
netflix_df["date_added"].head(10)

In [ ]:
# Sample non-missing dates to check formatting consistency
netflix_df["date_added"].dropna().sample(20, random_state=42)

In [ ]:
# Test whether all non-missing date values can be interpreted as valid dates
parsed_dates = pd.to_datetime(netflix_df["date_added"], errors="coerce")

parsed_dates.isna().sum()

### Investigating Director Name Variations

The initial consistency check identified director names that differ only in capitalization. These variations can cause the same director to be counted as separate individuals, so the affected records will be examined before standardizing the names.


In [ ]:
# Display the most frequent director names
netflix_df["director"].value_counts(dropna=False).head(20)

In [ ]:
netflix_df["director"].value_counts().unique()

In [ ]:
# Identify director names with leading or trailing whitespace
director_whitespace = (
    netflix_df["director"].dropna().astype(str).str.strip()
    != netflix_df["director"].dropna().astype(str)
)

director_whitespace.sum()

In [ ]:
# Find names that may differ only by capitalization
director_case_check = (
    netflix_df["director"]
    .dropna()
    .groupby(netflix_df["director"].dropna().str.lower())
    .filter(lambda x: x.nunique() > 1)
)

director_case_check.unique()

In [ ]:
# Display records containing the identified director name variations
director_variations = netflix_df[
    netflix_df["director"].isin([
        "Kevin Macdonald",
        "Kevin MacDonald",
        "Fernando Frías de la Parra",
        "Fernando Frías De La Parra"
    ])
]

director_variations[["title", "director"]]

In [ ]:
# Count each variation to determine the number of affected records
director_variations["director"].value_counts()

In [ ]:
# Standardize confirmed capitalization variations in director names
director_name_map = {
    "Kevin Macdonald": "Kevin MacDonald",
    "Fernando Frías de la Parra": "Fernando Frías De La Parra"
}

netflix_df["director"] = netflix_df["director"].replace(director_name_map)

In [ ]:
# Check that the inconsistent director names have been standardized
netflix_df[
    netflix_df["director"].isin([
        "Kevin MacDonald",
        "Fernando Frías De La Parra"
    ])
][["title", "director"]]

In [ ]:
# Identify columns containing the "Not Given" placeholder
not_given_counts = (netflix_df == "Not Given").sum()

not_given_counts[not_given_counts > 0]

In [ ]:
# Replace the source placeholder with a consistent missing-value label
netflix_df[["director", "country"]] = netflix_df[["director", "country"]].replace("Not Given", "N/A")

In [ ]:
# Confirm that "Not Given" no longer appears in these columns
(netflix_df[["director", "country"]] == "Not Given").sum()

In [ ]:
# Identify director names that still have capitalization variations
director_case_variations = (
    netflix_df["director"]
    .dropna()
    .groupby(netflix_df["director"].dropna().str.lower())
    .filter(lambda x: x.nunique() > 1)
)

director_case_variations.unique()

### Checking Country Naming Consistency

The `country` field may contain one or multiple countries for a title. The values will be examined for naming and formatting inconsistencies, including capitalization, spacing, and variations in how countries are represented.


In [ ]:
# Display the most frequent country entries
netflix_df["country"].value_counts(dropna=False).head(30)

In [ ]:
# Split multi-country entries into individual country names for consistency checking
country_names = (
    netflix_df["country"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

country_names.value_counts().head(40)

In [ ]:
# Identify country names that appear with different capitalization
country_case_variations = (
    country_names.groupby(country_names.str.lower())
    .filter(lambda x: x.nunique() > 1)
)

country_case_variations.unique()

In [ ]:
# Extract individual country names from multi-country entries
country_names = (
    netflix_df["country"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# Display all unique country names alphabetically
sorted(country_names.unique())

In [ ]:
# Check for individual country names with leading or trailing whitespace
country_whitespace = country_names[
    country_names != country_names.str.strip()
]

country_whitespace.unique()

#### Country Formatting Validation
**Finding:** The `country` field was checked for capitalization differences, alternative country names or aliases, and leading/trailing whitespace. No formatting or naming inconsistencies were identified. Multi-country entries were retained as valid values, while `Not Given` was identified as a missing-information placeholder.

**Action:** No formatting changes were required for the `country` field. `Not Given` will be addressed during the missing-value assessment.


### Checking listed_in Naming Consistency

The listed_in field contains one or more categories for each title. The category names will be reviewed for inconsistent capitalization, spacing, and duplicate representations that could affect category-level analysis.

In [ ]:
# Display the most frequent category combinations
netflix_df["listed_in"].value_counts().head(30)

In [ ]:
# Split multi-category entries into individual category names
listed_in_names = (
    netflix_df["listed_in"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# Display individual categories by frequency
listed_in_names.value_counts().head(30)

In [ ]:
# Identify categories that differ only by capitalization
listed_in_case_variations = (
    listed_in_names.groupby(listed_in_names.str.lower())
    .filter(lambda x: x.nunique() > 1)
)

listed_in_case_variations.unique()

In [ ]:
# Check for leading or trailing whitespace in category names
listed_in_whitespace = listed_in_names[
    listed_in_names != listed_in_names.str.strip()
]

listed_in_whitespace.unique()

#### Listed In Consistency Finding
**Finding**: The listed_in field was reviewed for capitalization inconsistencies and leading or trailing whitespace after separating multi-category entries. No formatting inconsistencies were identified. The different category names represent distinct content classifications rather than duplicate naming variations.

**Action**: No formatting changes were required for the listed_in field. The multi-category structure will be retained for subsequent analysis and feature engineering.

### Checking Duration Consistency

The duration field contains information about the length of both movies and TV shows. Because these content types use different units, the field will be examined for consistent formatting, valid duration values, and unexpected entries before transformation.


In [ ]:
# Display the most common duration values
netflix_df["duration"].value_counts(dropna=False).head(30)

In [ ]:
# Identify the measurement units used in the duration field
duration_units = (
    netflix_df["duration"]
    .dropna()
    .str.extract(r"([A-Za-z]+(?:\s[A-Za-z]+)?)")[0]
    .value_counts()
)

duration_units


In [ ]:
# Compare duration formats across movies and TV shows
pd.crosstab(
    netflix_df["type"],
    netflix_df["duration"].str.extract(r"([A-Za-z]+(?:\s[A-Za-z]+)?)")[0],
    dropna=False
)

#### Duration Consistency Findings
**Finding:** The duration field uses min for movies and Season/Seasons for TV shows. The cross-check with type confirmed that the units are structurally consistent with the content type. Since season counts cannot be reliably converted into viewing minutes without episode-level information, the two measures will be treated separately.

**Action:** No conversion between minutes and seasons will be performed. The duration field will be retained in its original form and may later be separated into a numeric value and measurement unit during feature engineering.


### Checking Title Consistency

The title field is important for identifying and analyzing individual content items. We’ll check for formatting issues such as leading/trailing whitespace, unusual capitalization, and repeated titles. Repeated titles are not automatically errors because different movies or shows can legitimately share the same name.

In [ ]:
# Check for leading or trailing whitespace in titles
title_whitespace = netflix_df["title"][
    netflix_df["title"] != netflix_df["title"].str.strip()
]

title_whitespace.unique()

In [ ]:
# Identify titles that appear more than once
duplicate_titles = netflix_df["title"].value_counts()

duplicate_titles[duplicate_titles > 1].head(20)

In [ ]:
# Remove leading/trailing whitespace, including non-breaking spaces
netflix_df["title"] = netflix_df["title"].str.replace("\xa0", " ", regex=False).str.strip()

In [ ]:
# Confirm that no titles still contain leading or trailing whitespace
title_whitespace = netflix_df["title"][
    netflix_df["title"] != netflix_df["title"].str.strip()
]

title_whitespace.unique()

In [ ]:
# Inspect records with repeated title values
netflix_df[netflix_df["title"].isin(["9-Feb", "15-Aug", "22-Jul"])]

**Finding:** The `title` field contained a non-breaking trailing space in `Consequences`, which was identified as a formatting inconsistency. The field also contained repeated values for `9-Feb`, `15-Aug`, and `22-Jul`. Inspection showed that each repeated value corresponded to identical records, indicating duplicate content records rather than separate titles with the same name.

**Action:** Whitespace formatting was standardized across the `title` field. The repeated records will be investigated further against the raw dataset before any records are removed or titles are modified.


In [ ]:
# Identify titles that match common day-month date patterns
date_like_titles = netflix_df[
    netflix_df["title"].str.match(r"^\d{1,2}-[A-Za-z]{3}$", na=False)
]

date_like_titles[["show_id", "title", "type", "date_added"]]

In [ ]:
import pandas as pd

# Load the original dataset separately for comparison
netflix_raw = pd.read_csv("Dataset.csv")

In [ ]:
netflix_raw.head()

In [ ]:
# Compare the suspicious show IDs with their original titles
netflix_raw[
    netflix_raw["show_id"].isin(
        ["s3997", "s5965", "s3963", "s4523", "s5964", "s5966", "s5967"]
    )
][["show_id", "title"]]

In [ ]:
# Inspect the complete raw records for the suspicious titles to confirm that we didn't alter the data when loading and preparing it
netflix_raw[
    netflix_raw["show_id"].isin(
        ["s3997", "s5965", "s3963", "s4523", "s5964", "s5966", "s5967"]
    )
].T

In [ ]:
# Check the original row positions of the suspicious records
netflix_raw[
    netflix_raw["show_id"].isin(
        ["s3997", "s5965", "s3963", "s4523", "s5964", "s5966", "s5967"]
    )
].index

In [ ]:
# After confirming that our original dataset had the same errors, we will not continue with our working dataset
# Inspect nearby rows to identify whether the suspicious titles follow a pattern
suspicious_indices = netflix_df[
    netflix_df["show_id"].isin(
        ["s3997", "s5965", "s3963", "s4523", "s5964", "s5966", "s5967"]
    )
].index

netflix_df.loc[
    sorted(set(i for idx in suspicious_indices for i in range(max(0, idx - 2), idx + 3))),
    ["show_id", "title", "type", "date_added"]
]

#### Title consistency observation
**Observation:** The title field was generally consistent, with one identified whitespace inconsistency in Consequences, which was standardized. Several titles appeared date-like, but inspection of the raw dataset and surrounding records confirmed that these were valid content titles. Repeated titles were identified separately and will be addressed during duplicate-record cleaning.

### Checking Release Year Validity

The release_year field is numeric, so instead of checking formatting, we want to identify unusual or potentially invalid years. We'll first establish the range before deciding whether anything needs investigation.

In [ ]:
# Inspect the distribution and range of release years
netflix_df["release_year"].describe()

In [ ]:
# Identify the earliest and latest release years
netflix_df["release_year"].agg(["min", "max"])

In [ ]:
# Inspect the oldest release years represented in the dataset
netflix_df["release_year"].sort_values().head(20).to_list()

In [ ]:
# Inspect the most recent release years in the dataset
netflix_df["release_year"].sort_values(ascending=False).head(20)

In [ ]:
# Identify records where the release year is later than the year the title was added
year_inconsistencies = netflix_df[
    netflix_df["release_year"] > netflix_df["date_added"].dt.year
]

year_inconsistencies[["title", "release_year", "date_added"]]

In [ ]:
netflix_df["release_year"].unique()

**Finding:** The `release_year` field contains values ranging from 1925 to 2021. The values fall within a plausible range for the content represented in the dataset, with no obvious invalid or impossible entries identified.

**Action:** No changes were required for the `release_year` field.


In [ ]:
# Check whether content type contains only expected categories
netflix_df["type"].unique()

### Step 4: Transform categorical and date-related columns.

#### Extracting Date Components

The date_added field has already been converted to datetime format. To make the information easier to analyze and use in downstream models, relevant date components such as year, month, and day will be extracted into separate features.

In [ ]:
# Extract useful components from the date the title was added
netflix_df["date_added_year"] = netflix_df["date_added"].dt.year
netflix_df["date_added_month"] = netflix_df["date_added"].dt.month
netflix_df["date_added_day"] = netflix_df["date_added"].dt.day

In [ ]:
# Verify the newly created date features
netflix_df[["date_added", "date_added_year", "date_added_month", "date_added_day"]].head(20)


#### Feature engineering 
To create new duration columns with values and strings separately 
since we are nit sure of minutes in each season so that we can have all as mins

In [ ]:
# Create separate numeric and unit features while preserving the original duration
netflix_df["duration_value"] = netflix_df["duration"].str.extract(r"(\d+)").astype(int)
netflix_df["duration_unit"] = netflix_df["duration"].str.extract(r"([A-Za-z]+)")

In [ ]:
# Standardize singular and plural season labels
netflix_df["duration_unit"] = netflix_df["duration_unit"].replace({
    "Seasons": "Season"
})

In [ ]:
# Check the transformed duration features
netflix_df[["type", "duration", "duration_value", "duration_unit"]].head(10)

In [ ]:
# Confirm the duration units and their distribution
netflix_df["duration_unit"].value_counts()

### Transforming Content Categories

The `listed_in` field contains one or more content categories separated by commas. The individual categories will be extracted so they can be analyzed as separate categorical features while preserving the original field.


In [ ]:
# Split the listed categories into individual values
genre_lists = netflix_df["listed_in"].str.split(", ")

# Inspect a few examples
genre_lists.head()

In [ ]:
# Count the number of categories assigned to each title
netflix_df["genre_count"] = genre_lists.str.len()

In [ ]:
# Check the distribution of categories assigned to each title
netflix_df["genre_count"].value_counts().sort_index()

#### Validating Transformed Features

The newly created features will be checked to confirm that the transformations were applied correctly and did not introduce unexpected missing values or inconsistencies.

In [ ]:
# Review the structure and data types after feature transformation 
netflix_df.info()

In [ ]:
# Check for missing values in the newly created features 
netflix_df[ ["date_added_year", "date_added_month", 
             "date_added_day", "duration_value", 
             "duration_unit", "genre_count"] ].isna().sum()

In [ ]:
# Confirm that the final dataset contains no exact duplicate records
netflix_df.duplicated().sum()

In [ ]:
# Preview the transformed dataset
netflix_df.head()

### Step 5: Create a clean dataset for further analysis

In [ ]:
# Save the cleaned and transformed dataset for use in downstream notebooks
netflix_df.to_csv("netflix_cleaned.csv", index=False)

## Data Wrangling and Dataset Preparation

The raw Netflix dataset was systematically assessed and prepared for downstream analysis and modeling. The process included data inspection, quality assessment, consistency checks, cleaning, transformation, and feature engineering.

#### Key preparation steps included:

1. Standardizing inconsistent text values and whitespace.
2. Validating categorical fields and checking for duplicate records.
3. Assessing missing values and handling source-level missing information codes.
4. Validating date and release-year fields for appropriate formats and plausible values.
5. Converting date_added to a datetime format and extracting year, month, and day features.
6. Separating duration into numeric values and measurement units while preserving the original field.
7. Extracting the number of categories associated with each title through the genre_count feature.
8. Preserving multi-value fields such as country and listed_in for use in downstream analysis.
9. Performing final validation to ensure the processed dataset is structurally consistent and ready for further analysis.

The resulting clean dataset provides a reliable foundation for exploratory analysis, recommendation modeling, trend forecasting, and classification tasks in the subsequent project stages.